# 실습 4 · Rocchio 질의 수정 — 유사 사례 검색

장표 22 · 23. 1차 검색 결과로 질의 벡터를 옮겨 다시 찾는다. 두 가지를 비교한다.

| 방식 | 옮기는 방향 | 현실의 모습 |
|---|---|---|
| **실제 피드백** (장표 22) | `q' = α·q + β·mean(관련) − γ·mean(무관)` · 관련 / 무관 = 사용자가 상위 결과에 표시 | 변호사가 판례 검색 결과 10건 중 "관련 있음"을 체크 → 다시 검색 |
| **의사 피드백 PRF** (장표 23) | `q' = α·q + β·mean(상위 k)` · 상위 k 를 전부 관련이라고 가정 | 사용자 표시 없이 자동으로 |

| 과제 | 산업 | 질의 → 찾을 것 | "관련" = 사용자가 표시할 것 |
|---|---|---|---|
| lbox | 법무 | 사건 사실관계 → 유사 판례 | 같은 죄명 |
| osha | 제조 · 건설 안전 | 사고 경위 → 유사 사고 | 같은 사고 유형 |
| nhtsa | 제조 · 품질 | 결함 신고 → 유사 신고 | 같은 결함 부품 |
| aviation | 항공 | 원인 요약 → 유사 사고 | 같은 원인 분류 |
| KLUE-MRC | 검색 (한국어) | 질문 → 정답 지문 | 정답 지문 1개 (4-1 실습1 의 `docs` · `questions`) |

- 사용자 표시는 라벨로 흉내 낸다 : 상위 `SHOWN`(10)건 중 질의와 라벨이 같은 것 = "관련" 체크
- **잔여 집합 채점** : 사용자가 이미 본 상위 10건은 채점에서 뺀다 → "다시 검색했을 때 새로 보이는 10건(11위~)" 중 관련 비율 = P@10. 모든 방식에 같게
- 이 노트북만으로 끝난다 : 과제마다 `SIZE` 개(라벨별 균등)만 임베딩(기본 과제당 약 550 · 강사 PC 단독 약 8분, 실습 3 을 먼저 했다면 겹치는 글은 캐시에서). 그중 `N_QUERY` 개가 질의

| 셀 | 하는 일 |
|---|---|
| 1 | 설정 |
| 2 | 준비 — DB · 임베딩 서버 · 도구 (01~04 공통) |
| 3 | 테이블 만들기 — `lab3.case_items` |
| 4 | 받기 · 표 |
| 5 | 적재 |
| 6 | 임베딩 |
| 7 | 불러오기 · 검색 · 채점 도구 |
| 8 | Rocchio SQL — 판례 한 건 : 1차 10건 → 관련 표시 → `q + avg(관련)` → 다시 검색 |
| 9 | 1차 검색(기준) — 상위 10건 중 관련 비율 · 11위~ 의 P@10 |
| 10 | 실제 피드백 — β · γ |
| 11 | 의사 피드백 — β · k |
| 12 | 1차가 빗나간 질의 — 실제 피드백 vs 의사 피드백 (쿼리 드리프트) |
| 13 | 의사 피드백 보강 — 확신도 조건부 · 점수 가중 · 공통 성분 제거 |
| 14 | KLUE-MRC 질문 검색 — 정답이 하나뿐일 때 의사 피드백 (+ 보강) |
| 15 | 정리 |

### 셀 1 · 설정
- `SIZE` : 과제마다 글 수(라벨별 균등) · `N_QUERY` : 그중 질의로 쓸 글 수 · `SHOWN` : 사용자가 보고 표시하는 1차 결과 수
- 셀 13 은 같은 DB 의 4-1 실습1 임베딩(`docs` · `questions`)을 쓴다. 없으면 그 셀만 건너뛴다

In [ ]:
# 셀 1 · 설정
DB_URL       = "postgresql://lab:lab@db:5432/lab"       # 4-1 도커의 DB
LMSTUDIO_URL = "http://host.docker.internal:1234/v1"    # LM Studio 서버 주소 (4-1 실습1 셀 1 과 같게)
EMBED_MODEL  = "text-embedding-qwen3-embedding-8b"      # LM Studio 에 로드된 임베딩 모델 이름
CACHE        = "/cache/hf/lab3"                         # 받은 원본 저장 위치 = 도커 볼륨 hfcache
BATCH_SIZE   = 16                                       # 한 번에 LM Studio 로 보내는 글 수
MAX_CHARS    = 1500                                     # 이보다 긴 글은 잘라서 넣는다
SEED         = 42
RESET        = False    # True : 이 실습의 테이블을 지우고 다시 만든다(표본 크기를 바꿨을 때). 이미 계산한 벡터는 캐시에서 돌아온다

SIZE    = {"lbox": 550, "osha": 550, "nhtsa": 550, "aviation": 550}   # 과제마다 글 수
N_QUERY = 100
SHOWN   = 10                         # 사용자가 보는 1차 결과 수 (피드백 재료 · 채점에서 뺌)
ALPHA   = 1.0
BETAS   = [0.25, 0.5, 0.75, 1.0]
GAMMAS  = [0.0, 0.25, 0.5]
KS      = [3, 5, 10]                 # 의사 피드백 : 상위 몇 건을 관련으로 가정하나

### 셀 2 · 준비 — DB · 임베딩 서버 · 도구
01~04 실습의 셀 2 는 모두 같다(각 실습을 따로 돌릴 수 있게 노트북마다 들어 있다).
- DB 연결(4-1 과 같은 pgvector DB, 스키마 `lab3`) · LM Studio 확인
- **임베딩 캐시** `lab3.emb_cache` : 키 = (모델 이름, 글의 md5) → 벡터. 같은 글은 한 번만 계산한다(예 : 실습 3 에서 계산한 판례는 실습 4 에서 다시 계산하지 않음). 실습 테이블을 지워도 남고, 지우려면 `TRUNCATE lab3.emb_cache`
- `embed_table(테이블)` : 4-1 실습1 셀 7 과 같은 배치 임베딩. ① 캐시에서 먼저 채우고 ② 남은 행을 `BATCH_SIZE` 개씩 LM Studio 로 → `UPDATE` + 캐시 저장. 멈췄다 다시 실행하면 빈 행부터
- `load(SQL)` : 마지막 열이 벡터인 SELECT → (표, 벡터 행렬 · 길이 1)
- `ci(값들)` : 단위별 차이의 평균과 부트스트랩 95% 신뢰구간 — 표본이 작아 차이가 우연인지 함께 본다(구간이 0 을 포함하면 "차이가 있다고 말할 수 없음")

In [ ]:
# 셀 2 · 준비
import json, time, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psycopg
import requests
from pgvector.psycopg import register_vector

plt.rcParams.update({"font.family": "NanumGothic", "axes.unicode_minus": False, "figure.dpi": 110})
pd.set_option("display.max_colwidth", 70)
C = Path(CACHE); (C / "raw").mkdir(parents=True, exist_ok=True)
HF = "https://huggingface.co/datasets"
rng = np.random.default_rng(SEED)

conn = psycopg.connect(DB_URL, autocommit=True)          # 문장마다 바로 반영 (여러 문장을 묶을 때만 conn.transaction())
register_vector(conn)
conn.execute("CREATE SCHEMA IF NOT EXISTS lab3")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.emb_cache (          -- 임베딩 캐시 (01~04 공용)
        model     text NOT NULL,                         -- 임베딩 모델 이름 (모델이 다르면 벡터도 다르다)
        text_md5  text NOT NULL,                         -- md5(글)
        embedding vector(4096) NOT NULL,
        PRIMARY KEY (model, text_md5))""")


def one(q, params=None):
    """SQL 결과의 첫 행 첫 값"""
    row = conn.execute(q, params).fetchone()
    return row[0] if row else None


def sql(q, params=None):
    """SQL 결과 → 표"""
    with conn.cursor() as cur:
        cur.execute(q, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


def unit(v):
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


def ci(x, n=1000):
    """단위(사용자 · 질의 · 글 …)별 값 x 의 평균과 부트스트랩 95% 신뢰구간. 구간이 0 을 포함하지 않으면 ✔ (차이가 우연이라고 보기 어려움)"""
    x = np.asarray(x, float); r = np.random.default_rng(SEED)
    b = [x[r.integers(0, len(x), len(x))].mean() for _ in range(n)]
    lo, hi = np.quantile(b, [.025, .975])
    return f"{x.mean():+.3f} [{lo:+.3f}, {hi:+.3f}]" + (" ✔" if lo > 0 or hi < 0 else "")


def load(q, params=None):
    """마지막 열이 벡터인 SELECT → (표, 벡터 행렬 float32 · 길이 1). 이진 전송이라 벡터가 바로 numpy 로 온다"""
    with conn.cursor(binary=True) as cur:
        cur.execute(q, params)
        rows = cur.fetchall()
        names = [c.name for c in cur.description]
    assert rows and all(r[-1] is not None for r in rows), "임베딩이 빈 행이 있다 → 셀 6(임베딩)을 끝까지"
    return pd.DataFrame([r[:-1] for r in rows], columns=names[:-1]), unit(np.stack([r[-1] for r in rows]).astype(np.float32))


def download(url, name):
    """볼륨에 없을 때만 받는다(.part 로 받고 끝나면 이름 변경)"""
    p = C / "raw" / name
    if not p.exists():
        t = time.time(); tmp = p.with_suffix(p.suffix + ".part")
        urllib.request.urlretrieve(url, tmp); tmp.rename(p)
        print(f"  받음 {name} {p.stat().st_size / 1e6:.1f}MB · {time.time() - t:.0f}s")
    return p


def parquet_urls(ds, config, split):
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet/{config}/{split}", timeout=60)
    r.raise_for_status(); return r.json()


def all_parquet_urls(ds, config="default"):
    """그 구성의 모든 분할(train · test …) parquet 주소"""
    r = requests.get(f"https://huggingface.co/api/datasets/{ds}/parquet", timeout=60); r.raise_for_status()
    return [u for us in r.json()[config].values() for u in us]


def clean(d):
    """글 다듬기 : NUL 제거 · 앞뒤 공백 · MAX_CHARS 에서 자르기"""
    d = d.reset_index(drop=True)
    d["text"] = d["text"].astype(str).str.replace("\x00", "", regex=False).str.strip().str.slice(0, MAX_CHARS)
    d["label"] = d["label"].astype(str)
    return d


def per_label(d, k):
    """라벨마다 k 개씩 무작위. 같은 SEED 면 작은 k 의 표본은 큰 k 표본의 앞부분이다(→ 캐시가 그대로 맞는다)"""
    return pd.concat([g.sample(min(len(g), k), random_state=SEED) for _, g in d.groupby("label")]).reset_index(drop=True)


def embed(texts):
    """LM Studio /v1/embeddings. 실패하면 5 · 10 · 20초 뒤 다시"""
    for wait in [5, 10, 20, None]:
        try:
            r = requests.post(f"{LMSTUDIO_URL}/embeddings", json={"model": EMBED_MODEL, "input": texts}, timeout=600)
            r.raise_for_status()
            return [np.asarray(d["embedding"], dtype=np.float32) for d in sorted(r.json()["data"], key=lambda d: d["index"])]
        except Exception as e:
            if wait is None: raise
            print(f"  ! 임베딩 실패({type(e).__name__}) → {wait}s 뒤 다시"); time.sleep(wait)


def embed_table(table):
    """빈 embedding 채우기 : ① 캐시(같은 모델 · 같은 글) → ② 남은 행을 BATCH_SIZE 개씩 LM Studio 로 · UPDATE + 캐시 저장"""
    t0 = time.time()
    hit = conn.execute(f"""UPDATE {table} t SET embedding = c.embedding FROM lab3.emb_cache c
                           WHERE t.embedding IS NULL AND c.model = %s AND c.text_md5 = md5(t.text)""", (EMBED_MODEL,)).rowcount
    todo = one(f"SELECT count(*) FROM {table} WHERE embedding IS NULL")
    print(f"[{table}] 캐시에서 {hit:,}건 · 새로 임베딩 {todo:,}건", flush=True)
    done, nb, t1 = 0, 0, time.time()
    while True:
        rows = conn.execute(f"SELECT id, text FROM {table} WHERE embedding IS NULL ORDER BY id LIMIT %s", (BATCH_SIZE,)).fetchall()
        if not rows:
            break
        vecs = embed([t for _, t in rows])
        with conn.transaction(), conn.cursor() as cur:
            cur.executemany(f"UPDATE {table} SET embedding = %s WHERE id = %s", [(v, i) for (i, _), v in zip(rows, vecs)])
            cur.executemany("INSERT INTO lab3.emb_cache VALUES (%s, md5(%s), %s) ON CONFLICT DO NOTHING",
                            [(EMBED_MODEL, t, v) for (_, t), v in zip(rows, vecs)])
        done += len(rows); nb += 1
        if nb % 25 == 0 or done >= todo:
            el = time.time() - t1; rate = done / el
            print(f"  {done:6,}/{todo:,}  {rate:5.1f}건/s  경과 {el / 60:5.1f}분  남은 시간 약 {max(todo - done, 0) / rate / 60:5.1f}분", flush=True)
    print(f"[{table}] 끝 · 캐시 {hit:,}건 + 새로 {done:,}건 · {time.time() - t0:.0f}s")


try:
    models = [m["id"] for m in requests.get(f"{LMSTUDIO_URL}/models", timeout=10).json()["data"]]
except requests.exceptions.RequestException:
    raise ConnectionError(f"LM Studio 에 연결할 수 없음 : {LMSTUDIO_URL}\n"
                          "  ① LM Studio 의 Developer 탭에서 서버가 켜져 있는지 확인\n"
                          "  ② 포트가 다르면 셀 1 의 LMSTUDIO_URL 을 고치고 셀 1 부터 다시") from None
assert EMBED_MODEL in models, f"셀 1 의 EMBED_MODEL 을 이 목록의 이름으로 바꾼다 : {models}"
print("pgvector", one("SELECT extversion FROM pg_extension WHERE extname = 'vector'"), "· 임베딩", embed(["확인"])[0].size, "차원",
      f"· 캐시 {one('SELECT count(*) FROM lab3.emb_cache WHERE model = %s', (EMBED_MODEL,)):,}건")

### 셀 3 · 테이블 만들기
`is_query` = 질의로 쓸 글(검색 대상에서 뺀다).

In [ ]:
# 셀 3 · 테이블
if RESET:
    conn.execute("DROP TABLE IF EXISTS lab3.case_items")
conn.execute("""
    CREATE TABLE IF NOT EXISTS lab3.case_items (
        id        bigserial PRIMARY KEY,
        ds        text    NOT NULL,              -- 과제 (lbox · osha · nhtsa · aviation)
        label     text    NOT NULL,              -- 관련 판정에 쓰는 라벨 (죄명 · 사고 유형 · 결함 부품 · 원인 분류)
        is_query  boolean NOT NULL,              -- 질의로 쓸 글
        text      text    NOT NULL,
        embedding vector(4096))""")
conn.execute("CREATE INDEX IF NOT EXISTS case_items_ds ON lab3.case_items (ds)")
sql("SELECT ds, count(*) AS 행 FROM lab3.case_items GROUP BY ds ORDER BY ds")

### 셀 4 · 받기 · 표
과제마다 원본을 받아(볼륨에 없을 때만) 라벨별 균등으로 `SIZE` 개 → 그중 무작위 `N_QUERY` 개를 질의로. 이미 테이블에 있는 과제는 건너뛴다.

In [ ]:
# 셀 4 · 받기 · 표
def lbox():
    """LBox Open : 한국 판례 사실관계 → 죄명 (많은 순 12종)"""
    d = pd.read_parquet(download(parquet_urls("lbox/lbox_open", "casename_classification_plus", "train")[0], "lbox_casename_plus_train.parquet"))
    top = d["casename"].value_counts().head(12).index
    return clean(d[d["casename"].isin(top)].assign(text=lambda x: x["facts"], label=lambda x: x["casename"])[["text", "label"]])


OSHA_EVENT = {"1": "폭행 · 동물", "2": "교통 · 운반기계", "3": "화재 · 폭발", "4": "추락 · 넘어짐",
              "5": "유해물질 · 환경 노출", "6": "물체 · 장비 접촉(끼임 · 맞음)", "7": "과도한 힘 · 신체 반응"}


def osha():
    """OSHA 중대재해 보고 : 제조 · 건설 업종 사고 경위 → 사고 대분류 7종(사고 코드 앞자리)"""
    d = pd.read_parquet(download(parquet_urls("masonmarker/osha-severe-injury-reports-enriched", "default", "data")[0], "osha.parquet"))
    d = d[d["naics_sector_name"].isin(["Manufacturing", "Construction"]) & d["final_narrative"].notna()]
    d = d.assign(text=d["final_narrative"].str.strip(), label=d["event_code"].astype(str).str[0].map(OSHA_EVENT))
    return clean(d[d["label"].notna() & (d["text"].str.len() > 30)][["text", "label"]])


def nhtsa():
    """NHTSA 차량 결함 신고 → 결함 부품 대분류 5종"""
    d = pd.concat(pd.read_parquet(download(u, f"nhtsa_{i}.parquet")) for i, u in enumerate(all_parquet_urls("vic35get/nhtsa_complaints_dataset")))
    d = d.drop_duplicates("summary").assign(text=lambda x: x["summary"].str.strip())
    return clean(d[d["text"].str.len() > 30][["text", "label"]])


def aviation():
    """항공 사고 원인 요약 → 원인 분류 (많은 순 8종)"""
    d = pd.read_parquet(download(parquet_urls("himaxym/aviation-safety-occurrences", "default", "train")[0], "aviation_0.parquet"))
    d = d[d["cause_summary"].fillna("").str.len() > 50]
    top = d["category"].value_counts().head(8).index
    return clean(d[d["category"].isin(top)].assign(text=lambda x: x["cause_summary"], label=lambda x: x["category"])[["text", "label"]])


have = set(sql("SELECT DISTINCT ds FROM lab3.case_items")["ds"]) if one("SELECT count(*) FROM lab3.case_items") else set()
S = {}
for ds, n in SIZE.items():
    if ds in have: print(f"{ds:9s} 테이블에 있음"); continue
    t = time.time()
    d = globals()[ds]()
    s = per_label(d, n // d["label"].nunique())
    q = np.random.default_rng(SEED).choice(len(s), N_QUERY, replace=False)
    S[ds] = s.assign(ds=ds, is_query=np.isin(np.arange(len(s)), q))
    print(f"{ds:9s} 라벨 {s['label'].nunique():2d} · 글 {len(s):,} (질의 {N_QUERY}) · {time.time() - t:.0f}s")

### 셀 5 · 적재

In [ ]:
# 셀 5 · 적재
for ds, d in S.items():
    conn.execute("""INSERT INTO lab3.case_items (ds, label, is_query, text)
                    SELECT * FROM unnest(%s::text[], %s::text[], %s::bool[], %s::text[])""",
                 [d["ds"].tolist(), d["label"].tolist(), d["is_query"].tolist(), d["text"].tolist()])
conn.execute("ANALYZE lab3.case_items")
sql("""SELECT ds, count(DISTINCT label) AS 라벨, count(*) FILTER (WHERE is_query) AS 질의, count(*) FILTER (WHERE NOT is_query) AS 검색_대상,
              percentile_disc(0.5) WITHIN GROUP (ORDER BY length(text)) AS 글_길이_중앙
       FROM lab3.case_items GROUP BY ds ORDER BY ds""")

### 셀 6 · 임베딩
실습 3 을 먼저 했다면 겹치는 글은 캐시에서 채워진다.

In [ ]:
# 셀 6 · 임베딩
embed_table("lab3.case_items")

### 셀 7 · 불러오기 · 검색 · 채점 도구
- `first(t)` : 1차 검색 상위 `SHOWN` 건과 관련 표시(라벨이 같은가)
- `p10(t, 질의들)` : 상위 `SHOWN` 건을 뺀 나머지에서 P@10 (질의마다)
- `feedback(…)` : 실제 피드백 `α·q + β·mean(관련) − γ·mean(무관)` · `prf(…)` : 의사 피드백 `α·q + β·mean(상위 k)` → 둘 다 길이 1 로
- 질의가 과제당 100개뿐이라 변화에는 셀 2 의 `ci()`(질의 단위 95% 신뢰구간)를 함께 붙인다

In [ ]:
# 셀 7 · 도구
T = {}
for ds in SIZE:
    d, X = load("SELECT id, label, is_query, embedding FROM lab3.case_items WHERE ds = %s ORDER BY id", (ds,))
    q = d["is_query"].values
    T[ds] = dict(Q=X[q], ql=d["label"].values[q], D=X[~q], dl=d["label"].values[~q], ids=d["id"].values[q])
print({k: f"질의 {len(v['Q'])} · 검색 대상 {len(v['D']):,}" for k, v in T.items()})


def first(t):
    top = np.argsort(-(t["Q"] @ t["D"].T), axis=1)[:, :SHOWN]
    return top, t["dl"][top] == t["ql"][:, None]           # (상위 SHOWN 번호, 관련 표시)


def p10(t, Q, top):
    S = Q @ t["D"].T
    S[np.arange(len(Q))[:, None], top] = -np.inf            # 이미 본 상위 SHOWN 건은 뺀다
    tt = np.argsort(-S, axis=1)[:, :10]
    return (t["dl"][tt] == t["ql"][:, None]).mean(axis=1)


def feedback(t, top, rel, beta, gamma, alpha=ALPHA):
    D, z = t["D"], np.zeros(t["D"].shape[1], np.float32)
    mr = np.stack([D[a[r]].mean(0) if r.any() else z for a, r in zip(top, rel)])      # 관련 평균 (없으면 0)
    mn = np.stack([D[a[~r]].mean(0) if (~r).any() else z for a, r in zip(top, rel)])  # 무관 평균 (없으면 0)
    return unit(alpha * t["Q"] + beta * mr - gamma * mn)


def prf(t, top, k, beta, alpha=ALPHA):
    return unit(alpha * t["Q"] + beta * t["D"][top[:, :k]].mean(axis=1))

### 셀 8 · Rocchio SQL — 판례 한 건
α = β = 1, γ = 0 인 실제 피드백 `q' = q + mean(관련)` 을 SQL 한 문장으로. 코사인 거리는 길이를 보지 않아 다시 길이 1 로 만들 필요가 없다.
- `shown` = 1차 상위 10건 · 사용자가 "관련"으로 체크한 것 = `label = 질의의 죄명`(흉내)
- 다시 검색할 때 이미 본 10건은 뺀다(`id NOT IN shown`) → 11위 이후에서 새로 보이는 10건
- 질의 = 판례 질의 중 1차 10건에 관련이 3~6건인 첫 건 · 세 칸 비교 : 원래 질의 그대로 · 의사 피드백(10건 전부) · 실제 피드백(체크한 것만)

In [ ]:
# 셀 8 · Rocchio SQL
top, rel = first(T["lbox"])
i = int(np.flatnonzero((rel.sum(1) >= 3) & (rel.sum(1) <= 6))[0])
arg = {"q": int(T["lbox"]["ids"][i]), "gold": T["lbox"]["ql"][i]}
SHOWN_SQL = """SELECT id, label, embedding FROM lab3.case_items WHERE ds = 'lbox' AND NOT is_query
               ORDER BY embedding <=> (SELECT embedding FROM lab3.case_items WHERE id = %(q)s) LIMIT 10"""


def again(new_q):
    """이미 본 10건(shown)을 빼고 new_q 벡터로 다시 검색한 10건의 죄명"""
    return sql(f"""
        WITH shown AS ({SHOWN_SQL})
        SELECT label FROM lab3.case_items WHERE ds = 'lbox' AND NOT is_query AND id NOT IN (SELECT id FROM shown)
        ORDER BY embedding <=> ({new_q}) LIMIT 10""", arg)["label"]


Q0 = "(SELECT embedding FROM lab3.case_items WHERE id = %(q)s)"
s0 = sql(SHOWN_SQL, arg)["label"]
print("질의 판례 죄명 :", arg["gold"], "·", one("SELECT left(text, 100) FROM lab3.case_items WHERE id = %s", (arg["q"],)), "…")
print(f"1차 10건 중 관련 {int(s0.eq(arg['gold']).sum())}건 :", " / ".join(("○" if l == arg["gold"] else "·") + l for l in s0))
cols = {"원래 질의 (11위~)": again(Q0),
        "의사 피드백 q + avg(10건)": again(f"{Q0} + (SELECT avg(embedding) FROM shown)"),
        "실제 피드백 q + avg(관련)": again(f"{Q0} + (SELECT avg(embedding) FROM shown WHERE label = %(gold)s)")}
display(pd.DataFrame({k: v.map(lambda l: ("○ " if l == arg["gold"] else "  ") + l) for k, v in cols.items()}))
print("P@10 :", " · ".join(f"{k} {v.eq(arg['gold']).mean():.1f}" for k, v in cols.items()))

### 셀 9 · 1차 검색(기준)
- 상위 10 관련 = 1차 결과 10건 중 관련 비율(사용자가 체크하는 건수 ÷ 10)
- 기준 P@10 = 원래 질의로 11위 이후 10건의 관련 비율 · 찍기 = 검색 대상 중 같은 라벨 비율

In [ ]:
# 셀 9 · 기준
R9, BASE = [], {}
for n, t in T.items():
    top, rel = first(t); BASE[n] = (top, rel, p10(t, t["Q"], top))
    R9.append({"과제": n, "상위 10 관련": rel.mean(), "기준 P@10 (11위~)": BASE[n][2].mean(),
               "찍기": np.mean([(t["dl"] == l).mean() for l in t["ql"]])})
R9 = pd.DataFrame(R9).set_index("과제").round(3)
R9

### 셀 10 · 실제 피드백 — β · γ
α = 1 고정. 관련 평균 쪽으로(β) · 무관 평균에서 멀리(γ). 표 = P@10, 마지막 열 = 기본값(β 0.75 · γ 0.25)의 기준 대비 변화와 신뢰구간.

In [ ]:
# 셀 10 · 실제 피드백
rows, R10 = [], []
for n, t in T.items():
    top, rel, p0 = BASE[n]
    for b in BETAS:
        for g in GAMMAS:
            rows.append({"과제": n, "β": b, "γ": g, "P@10": p10(t, feedback(t, top, rel, b, g), top).mean()})
    p = p10(t, feedback(t, top, rel, 0.75, 0.25), top)
    R10.append({"과제": n, "기준": p0.mean(), "실제 피드백 β0.75 γ0.25": p.mean(), "변화 [95% 구간]": ci(p - p0),
                "좋아진 질의": (p > p0).mean(), "나빠진 질의": (p < p0).mean()})
display(pd.DataFrame(rows).pivot_table(index=["과제", "γ"], columns="β", values="P@10").round(3))
R10 = pd.DataFrame(R10).set_index("과제").round(3)
R10

### 셀 11 · 의사 피드백 — β · k
사용자 표시 없이 상위 k 건을 전부 관련으로 가정. 같은 잔여 집합(11위~)으로 채점.

In [ ]:
# 셀 11 · 의사 피드백
rows, R11 = [], []
for n, t in T.items():
    top, rel, p0 = BASE[n]
    for k in KS:
        for b in BETAS:
            rows.append({"과제": n, "k": k, "β": b, "P@10": p10(t, prf(t, top, k, b), top).mean()})
    p = p10(t, prf(t, top, 5, 0.5), top)
    R11.append({"과제": n, "기준": p0.mean(), "의사 피드백 k5 β0.5": p.mean(), "변화 [95% 구간]": ci(p - p0),
                "좋아진 질의": (p > p0).mean(), "나빠진 질의": (p < p0).mean()})
display(pd.DataFrame(rows).pivot_table(index=["과제", "k"], columns="β", values="P@10").round(3))
R11 = pd.DataFrame(R11).set_index("과제").round(3)
R11

### 셀 12 · 1차가 빗나간 질의 — 쿼리 드리프트
장표 23 : 1차 상위가 빗나가면 의사 피드백은 그 오류 쪽으로 질의를 옮긴다. 실제 피드백은 무관을 밀어낸다.
1차 상위 10 관련 비율 구간별로 두 방식의 평균 변화(질의 수가 적은 구간은 흔들린다).

In [ ]:
# 셀 12 · 드리프트
rows = []
for n, t in T.items():
    top, rel, p0 = BASE[n]
    pf, pp = p10(t, feedback(t, top, rel, 0.75, 0.25), top), p10(t, prf(t, top, 5, 0.5), top)
    r1 = rel.mean(1)
    for lo, hi, lab in [(0, 0.35, "상위 10 관련 0~3건"), (0.35, 0.75, "4~7건"), (0.75, 1.01, "8~10건")]:
        m = (r1 >= lo) & (r1 < hi)
        if m.sum(): rows.append({"과제": n, "1차": lab, "질의 수": int(m.sum()), "기준 P@10": p0[m].mean(),
                                 "실제 피드백 변화": (pf - p0)[m].mean(), "의사 피드백 변화": (pp - p0)[m].mean()})
pd.DataFrame(rows).set_index(["과제", "1차"]).round(3)

### 셀 13 · 의사 피드백 보강
사람 표시 없이 의사 피드백을 덜 위험하게 · 더 쓸모 있게 만드는 세 가지. 같은 잔여 집합(11위~)으로 채점, 계수는 미리 고정.
| 보강 | 방법 |
|---|---|
| 확신도 조건부 | 1차 상위 5개 코사인 평균이 과제 안 질의의 중앙값 이상일 때만 의사 피드백(k 5 · β 0.5), 아니면 원래 질의 그대로 |
| 점수 가중 | 상위 5개를 똑같이 평균하지 않고 코사인 softmax(τ 0.05)로 가중 |
| 공통 성분 제거 | 검색 대상 전체 평균을 질의 · 문서에서 빼고 길이 1 → 1차 검색만 / 그 위에 의사 피드백 |
- 마지막 두 열 = 1차가 빗나간 질의(상위 10 관련 0~3건)에서의 변화 — 의사 피드백이 해를 끼치던 곳

In [ ]:
# 셀 13 · 의사 피드백 보강
def p10_on(t, Q, D, top):
    S = Q @ D.T; S[np.arange(len(Q))[:, None], top] = -np.inf
    tt = np.argsort(-S, axis=1)[:, :10]
    return (t["dl"][tt] == t["ql"][:, None]).mean(axis=1)


def prf_w(Q, D, S, k=5, beta=0.5):
    top = np.argsort(-S, axis=1)[:, :k]
    w = np.exp(np.take_along_axis(S, top, 1) / 0.05); w /= w.sum(1, keepdims=True)
    return unit(Q + beta * (D[top] * w[:, :, None]).sum(1))


rows = []
for n, t in T.items():
    top, rel, p0 = BASE[n]
    Q, D = t["Q"], t["D"]; S = Q @ D.T
    plain = prf(t, top, 5, 0.5)
    conf = np.sort(S, axis=1)[:, -5:].mean(1); gate = conf >= np.median(conf)
    mu = D.mean(axis=0); Qc, Dc = unit(Q - mu), unit(D - mu)
    topc = np.argsort(-(Qc @ Dc.T), axis=1)[:, :5]
    V = {"의사 피드백 그대로": p10(t, plain, top), "확신도 조건부": p10(t, np.where(gate[:, None], plain, Q), top),
         "점수 가중": p10(t, prf_w(Q, D, S), top), "공통 성분 제거 · 1차만": p10_on(t, Qc, Dc, top),
         "공통 성분 제거 + 의사 피드백": p10_on(t, unit(Qc + 0.5 * Dc[topc].mean(1)), Dc, top)}
    bad = rel.sum(1) <= 3
    for name, v in V.items():
        rows.append({"과제": n, "방식": name, "변화 [95% 구간]": ci(v - p0), "빗나간 질의 변화": ci((v - p0)[bad]) if bad.sum() > 1 else "-"})
R13 = pd.DataFrame(rows)
display(R13.pivot(index="방식", columns="과제", values="변화 [95% 구간]"))
R13.pivot(index="방식", columns="과제", values="빗나간 질의 변화")

### 셀 14 · KLUE-MRC — 정답이 하나뿐일 때 의사 피드백
진짜 "질문 → 정답 지문" 검색(4-1 실습1 의 원본 4096 임베딩, 평가 질문 500개 = 4-1 과 같은 질문). 정답이 지문 1개뿐이라 상위 k 는 대부분 무관 → 의사 피드백이 질의를 정답에서 떼어 놓는지 본다.
- 채점은 4-1 과 같이 전체 순위의 Hit@10 · MRR (정답 1개라 잔여 집합을 쓰지 않음) · 4-1 실습1 을 안 했으면 건너뛴다
- 아래 표 끝에 셀 13 의 보강(확신도 조건부 · 공통 성분 제거)도 같이

In [ ]:
# 셀 14 · KLUE-MRC
if one("SELECT to_regclass('public.docs')") and one("SELECT count(embedding) FROM docs"):
    docs = conn.execute("SELECT id, embedding FROM docs WHERE embedding IS NOT NULL ORDER BY id", binary=True).fetchall()
    qs = conn.execute("SELECT doc_id, embedding FROM questions WHERE id % 11 = 0 AND embedding IS NOT NULL ORDER BY id LIMIT 500", binary=True).fetchall()
    ids = np.array([r[0] for r in docs]); Dk = unit(np.stack([r[1] for r in docs]).astype(np.float32))
    gold = np.array([np.flatnonzero(ids == r[0])[0] for r in qs]); Qk = unit(np.stack([r[1] for r in qs]).astype(np.float32))
    topk = np.argsort(-(Qk @ Dk.T), axis=1)

    def hits(Q):
        S = Q @ Dk.T; r = (S > S[np.arange(len(Q)), gold][:, None]).sum(1) + 1
        return (r <= 10).astype(float), np.where(r <= 10, 1 / r, 0)

    h0, m0 = hits(Qk)
    rows = [{"k": "-", "β": "-", "Hit@10": h0.mean(), "MRR": m0.mean(), "Hit@10 변화": "", "MRR 변화": ""}]
    for k in [1, 3, 5]:
        for b in [0.25, 0.5, 1.0]:
            h, m = hits(unit(Qk + b * Dk[topk[:, :k]].mean(1)))
            rows.append({"k": k, "β": b, "Hit@10": h.mean(), "MRR": m.mean(), "Hit@10 변화": ci(h - h0), "MRR 변화": ci(m - m0)})
    S = Qk @ Dk.T; conf = np.sort(S, axis=1)[:, -5:].mean(1); gate = conf >= np.median(conf)
    mu = Dk.mean(axis=0); Qc, Dc = unit(Qk - mu), unit(Dk - mu)
    hitsc = lambda Q: (lambda S2: (lambda r: ((r <= 10).astype(float), np.where(r <= 10, 1 / r, 0)))((S2 > S2[np.arange(len(Q)), gold][:, None]).sum(1) + 1))(Q @ Dc.T)
    for name, (h, m) in [("확신도 조건부 k1 β1.0", hits(np.where(gate[:, None], unit(Qk + 1.0 * Dk[topk[:, :1]].mean(1)), Qk))),
                         ("공통 성분 제거 · 1차만", hitsc(Qc)),
                         ("공통 성분 제거 + k5 β0.5", hitsc(unit(Qc + 0.5 * Dc[np.argsort(-(Qc @ Dc.T), axis=1)[:, :5]].mean(1))))]:
        rows.append({"k": name, "β": "", "Hit@10": h.mean(), "MRR": m.mean(), "Hit@10 변화": ci(h - h0), "MRR 변화": ci(m - m0)})
    R14 = pd.DataFrame(rows).round(3); display(R14)
else:
    print("4-1 실습1 의 임베딩(docs)이 없어 건너뜀")

### 셀 15 · 정리
- 셀 10 : 사용자가 관련을 표시하면(실제 피드백) 다시 검색한 결과가 좋아지는가 · 셀 11 : 표시 없이(의사 피드백)도 그런가
- 셀 12 : 1차가 빗나간 질의에서 두 방식이 어떻게 다른가
- 셀 13 : 확신도 조건부는 빗나간 질의의 손해를 막지만 이득도 줄인다 · 공통 성분 제거 + 의사 피드백은 이득이 커진다(과제에 따라) · 점수 가중은 차이 없음
- 셀 14 : 정답이 하나뿐인 검색에서는 보강해도 의사 피드백이 이득이 없다

In [ ]:
# 셀 15 · 정리
display(R9)
display(R10)
display(R11)
R13.pivot(index="방식", columns="과제", values="변화 [95% 구간]")